In [1]:
from pathlib import Path
import pandas as pd

In [2]:
BUILD_STAGE1_PY = Path("./check0.py")
!python {BUILD_STAGE1_PY}

saved: ../../DATA/dataset/CHECK_stage1
shape: (166904, 12)
n_features: 10
mem(MB): 5.41


In [3]:
import importlib.util

py_path = Path("data1.py")
spec = importlib.util.spec_from_file_location("data1", py_path)
data1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data1)

In [4]:
data1.build_train_stage2(
    in_stage2_path="../../DATA/dataset/check_stage2",
    stage1_path="../../DATA/dataset/CHECK_stage1",
    out_stage2_path="../../DATA/dataset/CHECK_stage2",
)

saved: ../../DATA/dataset/CHECK_stage2
shape: (166523, 24)
n_features: 22
mem(MB): 10.64
stage1_added_cols: 10
stage2_cols: 12


---

### drift_report_train_vs_check.py

In [1]:
!python drift_report_train_vs_check.py

saved: ../../../DATA/drift_reports/drift_train_vs_check.csv
saved: ../../../DATA/drift_reports/drift_train_vs_check.parquet
    n_train  n_check  ... flag_ks_ge_0_15 flag_rate_diff_ge_0_03
0    608430   166523  ...            True                  False
1    608430   166523  ...            True                  False
2    608430   166523  ...           False                   True
3    608430   166523  ...           False                  False
4    608430   166523  ...           False                   True
5    608430   166523  ...           False                  False
6    608430   166523  ...           False                  False
7    608430   166523  ...           False                  False
8    608430   166523  ...           False                  False
9    608430   166523  ...           False                  False
10   608430   166523  ...           False                  False
11   608430   166523  ...           False                  False
12   608430   166523  ...      

In [4]:
import pandas as pd
report = pd.read_parquet("../../../DATA/drift_reports/drift_train_vs_check.parquet")
report.head(50)

,n_train,n_check,feature,kind,psi,ks,train_missing,check_missing,train_mean,check_mean,...,train_p10,check_p10,train_p90,check_p90,train_p95,check_p95,flag_psi_ge_0_25,flag_psi_ge_0_10,flag_ks_ge_0_15,flag_rate_diff_ge_0_03
0,608430,166523,mcc_smoothed_risk,numeric,1.805288e+00,0.475516,0.0,0.0,0.005273,0.007318,...,0.000042,0.000357,1.949054e-02,0.027571,2.427527e-02,3.738382e-02,True,True,True,False
1,608430,166523,month_sin,numeric,1.018773e-01,0.164017,0.0,0.0,-0.008943,0.204143,...,-0.866025,-0.866025,8.660254e-01,1.000000,1.000000e+00,1.000000e+00,False,True,True,False
2,608430,166523,is_risky_month,binary,6.082378e-02,0.110854,0.0,0.0,0.340966,0.230112,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True
3,608430,166523,log_interval_dev,numeric,4.037524e-02,0.077190,0.0,0.0,0.114739,0.442496,...,-2.115744,-1.877278,1.647371e+00,1.968140,2.004927e+00,2.598493e+00,False,False,False,False
4,608430,166523,client_mcc_is_new,binary,2.860291e-02,0.034203,0.0,0.0,0.027885,0.062088,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,True
5,608430,166523,merchant_change_last5_x_amount_limit_ratio,numeric,4.649888e-03,0.028975,0.0,0.0,0.113919,0.093292,...,0.000000,0.000000,8.276041e-02,0.071332,3.941762e-01,3.348803e-01,False,False,False,False
6,608430,166523,seconds_since_prev_tx,numeric,4.223931e-03,0.031771,0.0,0.0,358867.076167,323912.507582,...,5220.000000,4200.000000,1.017060e+06,913908.000000,1.646853e+06,1.510776e+06,False,False,False,False
7,608430,166523,is_refund,binary,4.153821e-03,0.003810,0.0,0.0,0.005749,0.001940,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False
8,608430,166523,is_risky_wd_hour,binary,3.533598e-03,0.019511,0.0,0.0,0.113318,0.132828,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False
9,608430,166523,amount_limit_ratio,numeric,2.621719e-03,0.015235,0.0,0.0,0.047292,0.045637,...,0.000728,0.000746,3.371857e-02,0.031902,3.108656e-01,3.359941e-01,False,False,False,False


In [ ]:
# “경고 후보”만 필터링 셀
cand = report[
    report["flag_psi_ge_0_25"] |
    report["flag_ks_ge_0_15"] |
    report.get("flag_rate_diff_ge_0_03", False)
].copy()

cand[[
    "feature","kind","psi","ks","train_missing","check_missing",
    "train_mean","check_mean","train_p50","check_p50","train_p99","check_p99",
    "flag_psi_ge_0_25","flag_ks_ge_0_15"
]].head(100)

,feature,kind,psi,ks,train_missing,check_missing,train_mean,check_mean,train_p50,check_p50,train_p99,check_p99,flag_psi_ge_0_25,flag_ks_ge_0_15
0,mcc_smoothed_risk,numeric,1.805288,0.475516,0.0,0.0,0.005273,0.007318,7.687808e-04,0.002918,0.039785,0.037384,True,True
1,month_sin,numeric,0.101877,0.164017,0.0,0.0,-0.008943,0.204143,-2.449294e-16,0.500000,1.000000,1.000000,False,True
2,is_risky_month,binary,0.060824,0.110854,0.0,0.0,0.340966,0.230112,0.000000e+00,0.000000,1.000000,1.000000,False,False
4,client_mcc_is_new,binary,0.028603,0.034203,0.0,0.0,0.027885,0.062088,0.000000e+00,0.000000,1.000000,1.000000,False,False


In [ ]:
# “binary만” 요약 셀
b = report[report["kind"] == "binary"].copy()
b[["feature","psi","ks","train_rate_1","check_rate_1","abs_rate_diff","flag_rate_diff_ge_0_03"]].sort_values(
    ["abs_rate_diff","psi"], ascending=[False, False]
).head(50)

,feature,psi,ks,train_rate_1,check_rate_1,abs_rate_diff,flag_rate_diff_ge_0_03
2,is_risky_month,6.082378e-02,0.110854,0.340966,0.230112,0.110854,True
4,client_mcc_is_new,2.860291e-02,0.034203,0.027885,0.062088,0.034203,True
8,is_risky_wd_hour,3.533598e-03,0.019511,0.113318,0.132828,0.019511,False
7,is_refund,4.153821e-03,0.003810,0.005749,0.001940,0.003810,False
15,err_bad_cvv,1.285741e-05,0.000224,0.004025,0.003801,0.000224,False
16,limit_ratio_extreme,3.712436e-09,0.000002,0.001001,0.001003,0.000002,False
21,vel_x_mcc_risk,0.000000e+00,0.000000,0.000000,0.000000,0.000000,False


## TRAIN vs CHECK drift 리포트 해석 방법

### 지표 의미

| 지표                          | 의미                       | 해석 포인트                                                                     |
| --------------------------- | ------------------------ | -------------------------------------------------------------------------- |
| PSI                         | 분포가 “얼마나” 달라졌는지(빈 기반 거리) | 분포 형태 변화에 민감. 값이 크면 covariate shift 가능성 증가                                 |
| KS                          | 두 분포의 누적분포(CDF) 최대 차이    | 형태가 달라졌는지 빠르게 확인. 표본이 크면 민감하게 반응                                           |
| train_mean / check_mean     | 평균 이동                    | 전체 mass가 좌/우로 이동했는지                                                        |
| train_p50 / check_p50       | 중앙값 이동                   | typical 구간 이동 여부                                                           |
| train_p99 / check_p99       | 상위 꼬리 이동                 | tail risk 구간 변화 여부                                                         |
| train_rate_1 / check_rate_1 | 이진/플래그 피처의 1 비율          | 룰/플래그 활성화 수준이 바뀌었는지                                                        |
| abs_rate_diff               | 이진 비율 절대차                | 운영 조건 변화(룰 firing 빈도 변화) 판단                                                |
| flags                       | 빠른 경보                    | `flag_psi_ge_0_25`, `flag_ks_ge_0_15`, `flag_rate_diff_ge_0_03` 중심으로 후보 선정 |

### 기준 세팅

| 기준                   | 주의           | 강한 drift          |
| -------------------- | ------------ | ----------------- |
| PSI                  | 0.10 이상      | 0.25 이상           |
| KS                   | 0.15 이상      | 0.25 이상(상대적으로 강함) |
| binary abs_rate_diff | 0.03 이상(3%p) | 0.05 이상(5%p)      |

---

## 현재 결과 요약(핵심 drift 후보)

### 상위 drift 후보(PSI/KS/비율 변화 기준)

| feature           | kind    |    PSI |     KS | train_mean/rate | check_mean/rate | 추가 관찰 포인트                     | 결론                                              |
| ----------------- | ------- | -----: | -----: | --------------: | --------------: | ----------------------------- | ----------------------------------------------- |
| mcc_smoothed_risk | numeric |  1.805 |  0.476 |    mean 0.00527 |    mean 0.00732 | p50 0.00077→0.00292로 중앙 이동이 큼 | 최우선 drift. MCC mix 변화 또는 MCC risk map 불일치 가능성 큼 |
| month_sin         | numeric |  0.102 |  0.164 |   mean -0.00894 |    mean 0.20414 | 월(시즌) 분포 차이를 시사               | 기간/시즌성 차이 가능성. ‘문제 drift’인지 별도 판단 필요            |
| is_risky_month    | binary  |  0.061 |  0.111 |      rate 0.341 |      rate 0.230 | 11.1%p 감소                     | 월 기반 룰 firing 빈도 변화. 룰 재정의/비활성화 검토 후보           |
| client_mcc_is_new | binary  | 0.0286 | 0.0342 |     rate 0.0279 |     rate 0.0621 | 3.42%p 증가(약 2배)               | novelty(신규 MCC 경험) 증가. 데이터 환경 변화 가능성            |

---

## 해석의 결론 구조(라벨이 모두 0인 CHECK 기준)

### 전제

* CHECK의 `fraud`가 전부 0이면 성능(precision/recall/lift)으로 drift 영향 판단은 불가
* 따라서 결론은 “입력 분포 변화(covariate shift) 관찰” 기반으로 내린다
* 반영 대상은 모델 자체보다 “아티팩트/룰/피처 정의” 쪽이 우선이다

### 현재 drift 상황 결론

* 가장 큰 분포 이동은 `mcc_smoothed_risk`에서 발생했으며(PSI 1.805, KS 0.476), CHECK 환경에서 MCC 기반 위험 스코어의 mass가 오른쪽으로 크게 이동했다. 이는 MCC 구성(mix) 변화 또는 기존 smoothed risk map의 적용 불일치를 시사한다.
* `month_sin`의 KS(0.164)와 평균 이동은 데이터 기간/시즌성 차이에 의해 발생했을 가능성이 크다. 이는 자연스러운 분포 차이일 수 있으나, 월 기반 룰(`is_risky_month`)의 firing 비율이 11.1%p 감소하여 룰 정의가 CHECK 환경에서 덜 활성화되고 있다.
* `client_mcc_is_new` 비율이 약 2배 증가하여(2.79%→6.21%), 고객의 MCC novelty 행동 패턴이 달라졌을 가능성이 있다.

---



## 결론

CHECK는 라벨 기반 성능 검증이 불가능하므로 입력 분포 기반 drift 관찰 결과로만 판단한다.

mcc_smoothed_risk에서 매우 강한 covariate shift가 확인되었으며, 이는 Stage1의 MCC risk map을 CHECK 환경을 반영하도록 혼합 재추정(artifact 업데이트 1순위) 할 필요성을 시사한다.

월/시즌 분포 차이가 존재하며(month_sin, is_risky_month), 이는 데이터 기간 차이에 기인했을 가능성이 있다. 따라서 모델 구조 변경보다는 월 기반 룰의 환경 적합성 점검 및 재선정 여부 검토가 우선이다.

novelty 관련 피처(client_mcc_is_new) 비율이 증가하여 고객 행동 패턴 변화 가능성이 관찰되었으며, novelty 블록에 대해 zero-mass 기반 추가 drift 분석 후 반영 여부 판단이 필요하다.

---
